# 05 - Check merged h5ad

Sanity-check the merged objects. obs consistency, cells per `data_status` / `dataset_id`, gene counts, shared genes, and how missing values entered via the outer join. **No analysis yet.**

In [ ]:
import sys
from pathlib import Path

# locate the v2 project root (folder containing config/dataset_manifest.yaml)
ROOT = Path.cwd()
while not (ROOT / "config" / "dataset_manifest.yaml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
print("project root:", ROOT)

import manifest_utils as mf
man = mf.load_manifest()
paths = mf.project_paths(ROOT)
mf.ensure_dirs(paths)

import anndata as ad
import pandas as pd

merged = {p.stem: ad.read_h5ad(p) for p in sorted(paths['merged'].glob('*.h5ad'))}
list(merged.keys())

In [ ]:
for name, a in merged.items():
    print(f'=== {name} ===  {a.n_obs} cells x {a.n_vars} genes')
    print('obs columns:', list(a.obs.columns))
    print()

## Cells by data_status and dataset_id

In [ ]:
for name, a in merged.items():
    print(f'=== {name} ===')
    if 'data_status' in a.obs:
        print(a.obs['data_status'].value_counts())
    print(a.obs['dataset_id'].value_counts())
    print()

## Gene counts and missingness (outer join introduces zeros/NaN)

In [ ]:
import numpy as np
import scipy.sparse as sp
for name, a in merged.items():
    X = a.X
    nnz = X.nnz if sp.issparse(X) else int(np.count_nonzero(X))
    print(f'{name}: {a.n_vars} genes, density={nnz / (a.n_obs * a.n_vars):.4f}')

## obs missingness per column

In [ ]:
for name, a in merged.items():
    print(f'=== {name} ===')
    miss = (a.obs.astype(str) == 'nan').sum() + a.obs.isna().sum()
    print(miss[miss > 0].sort_values(ascending=False).head(20))
    print()

Done. Analysis / QC / normalization / clustering / scVI are intentionally **not** part of this stage.